# Example 11.3.2 / 1: Compare the Model View with the Solver View

Companion notebook for **11.3.2 Substitution and Expression Expansion** in the AMPL lab Documentation, section "Compare the Model View with the Solver View". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare the Model View with the Solver View

AMPL keeps two views of every problem. The model view is what you declared; the solver view is
what remains after presolve and substitutions, and it is what the solver receives. Two pairs of
generic names count them, read in amplpy with `get_value()`:

| Name | Counts |
|---|---|
| `_nvars`, `_ncons` | variables and constraints in the model view |
| `_snvars`, `_sncons` | variables and constraints in the solver view |

`expand` prints a constraint as declared, with data substituted; `solexpand` prints it as the
solver receives it. They differ even for a linear row: here presolve fixes `x` at 1 from its
bounds, so `solexpand` shows `y >= 2` rather than `x + y >= 3`. The counts go further: the solver
receives one variable and no constraints, because presolve turns that one-variable row into a
bound on `y`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 1, <= 1;  # bounds fix x at 1
var y >= 0;        # the only variable presolve leaves free

minimize Use: y;  # use as little y as possible

subject to Need: x + y >= 3;  # presolve sends y >= 2 instead

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
print(ampl.get_output("expand Need; solexpand Need;"), end="")
print([ampl.get_value(n) for n in ("_nvars", "_ncons", "_snvars", "_sncons")])

subject to Need:
	x + y >= 3;

subject to Need:
	y >= 2;

[2, 1, 1, 0]


### Pure AMPL: model.run

Pure AMPL:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
expand Need;
solexpand Need;
display _nvars, _ncons, _snvars, _sncons;

subject to Need:
	x + y >= 3;

subject to Need:
	y >= 2;

_nvars = 2
_ncons = 1
_snvars = 1
_sncons = 0

